In [2]:
import pandas as pd

df_historique = pd.read_csv('../data/processed/historique_clean.csv')
df_pacerelle = pd.read_csv('../data/processed/parcelles_clean.csv')

#print(df_historique.head())

#print(df_historique.iloc[0])

#print(df_pacerelle.iloc[0])

#print(df_historique["millesime"])





In [3]:
df_propre = df_pacerelle[df_pacerelle["rendement_nul_a_verifier"] == 0]
print(df_propre.corr(numeric_only=True)["rendement_kg_ha"].sort_values())


pente_pct                  -0.343326
age_vignes_ans             -0.218279
altitude_m                 -0.207252
nord_sud_km                -0.089269
traitements_an             -0.012559
surface_ha                  0.004368
est_ouest_km                0.066860
densite_pieds_ha            0.067698
pluviometrie_mm             0.382574
azote_apporte_g_ha          0.584834
rendement_kg_ha             1.000000
rendement_nul_a_verifier         NaN
Name: rendement_kg_ha, dtype: float64


- Analyser mes colonnes 

In [4]:
print(df_propre.columns)

Index(['parcelle_id', 'surface_ha', 'cepage', 'age_vignes_ans', 'type_sol',
       'altitude_m', 'pente_pct', 'exposition', 'densite_pieds_ha',
       'pluviometrie_mm', 'azote_apporte_g_ha', 'traitements_an',
       'est_ouest_km', 'nord_sud_km', 'rendement_kg_ha', 'classement_recolte',
       'rendement_nul_a_verifier'],
      dtype='str')


Verdict de la question 1 :

Le chef de culture a un peu raison : le sol joue, mais ce n'est pas tout.
L'œnologue a peu raison : l'âge des vignes a un lien faible (-0,22).
Ce qui compte le plus, c'est l'azote apporté (+0,58), puis la pluie (+0,38) et la pente (-0,34).

In [5]:
df_propre.groupby("type_sol")["rendement_kg_ha"].mean()

type_sol
argilo-calcaire    6715.739130
caillouteux        6996.340000
limoneux           6503.536585
sableux            6579.385965
Name: rendement_kg_ha, dtype: float64

- Explorer mon historisation

In [11]:
print(df_historique.columns)
print(df_historique.shape)
print(df_historique.head(10))

Index(['parcelle_id', 'millesime', 'rendement_kg_ha', 'prix_kg_eur',
       'rendement_nul_a_verifier'],
      dtype='str')
(1338, 5)
  parcelle_id  millesime  rendement_kg_ha  prix_kg_eur  \
0        P150       2019             6874         2.02   
1        P150       2020             8072         1.83   
2        P150       2021             5036         2.26   
3        P150       2022             7642         1.58   
4        P150       2023             7265         1.10   
5        P150       2024             6911         1.96   
6         P75       2019             6550         1.92   
7         P75       2020             7954         2.38   
8         P75       2021             4755         1.37   
9         P75       2022             7533         1.67   

   rendement_nul_a_verifier  
0                     False  
1                     False  
2                     False  
3                     False  
4                     False  
5                     False  
6                

- Fire une comparaison de mon année avce l'année suivante, d'où "df_historique.groupby("parcelle_id")["rendement_kg_ha"].shift(-1)""

In [12]:
df_historique = df_historique.sort_values(["parcelle_id", "millesime"])

df_historique["rendement_N+1"] = (
    df_historique.groupby("parcelle_id")["rendement_kg_ha"].shift(-1)
)

df_historique.head(7)

,parcelle_id,millesime,rendement_kg_ha,prix_kg_eur,rendement_nul_a_verifier,rendement_N+1
1200,P10,2019,7514,1.92,False,8502.0
1201,P10,2020,8502,1.91,False,6066.0
1202,P10,2021,6066,1.21,False,9537.0
1203,P10,2022,9537,1.45,False,7086.0
1204,P10,2023,7086,2.53,False,7388.0
1205,P10,2024,7388,1.37,False,NaN
750,P100,2019,7178,1.56,False,7870.0


- La corrélation entre le rendement d'une année et celui de l'année suivante est de 0.44 : le lien est positif mais moyen. Une parcelle qui a bien produit a tendance à bien produire l'année suivante

In [13]:
print(df_historique["rendement_kg_ha"].corr(df_historique["rendement_N+1"]))

0.43573776796022967
